
# Практические работы № 11–12. Разработка и защита проекта геопространственного анализа

**Дисциплина:** Анализ данных и машинное обучение (АДиМО).

**Команда:** № 1.

**Тема:** Здравоохранение и продолжительность жизни в субъектах Российской Федерации.

## 1. Постановка задачи исследования

### 1.1. Цель исследования

**Цель:** Исследование статистической и пространственной взаимосвязи между обеспеченностью населения врачами и ожидаемой продолжительностью жизни при рождении в субъектах Российской Федерации за 2021 год.

**Исследуемые показатели:**

- **Зависимая переменная $Y$:** Ожидаемая продолжительность жизни при рождении, измеряемая в годах, код Росстата `01160100`.
- **Независимая переменная $X$:** Численность врачей всех специальностей на 10 000 человек населения, код Росстата `05040200`.

**Основные методы:** Разведочный анализ данных (EDA), тематическое картографирование, обычная линейная регрессия (Ordinary Least Squares, OLS) и пространственная авторегрессионная модель (Spatial Autoregressive Model, SAR).

**Исследовательский вопрос:** Существует ли статистическая связь между обеспеченностью врачами и ожидаемой продолжительностью жизни, и как учёт пространственной зависимости может изменить оценку этой взаимосвязи?

### 1.2. Исходная геопространственная база

**Источник статистических показателей:** [Репозиторий regions_collection](https://github.com/tochno-st/regions_collection).

**Год наблюдений:** 2021.

**Единица наблюдения:** Субъект Российской Федерации.

**Подготовленные файлы:**

- `regions_2021_attributes.csv` — статистические показатели и территориальные идентификаторы.
- `regions_2021_geospatial.geojson` — географические границы регионов и сопутствующие атрибуты.

**Размерность:** 85 территориальных наблюдений с двумя количественными показателями.

**Система координат:** EPSG:4326 (WGS 84).

**Ключ объединения:** Код ОКАТО `object_okato`.

**Особенность:** ОКАТО уникален для всех 85 наблюдений, тогда как ОКТМО содержит повторяющееся значение для отдельных территорий, поэтому объединение выполняется по ОКАТО.

### 1.3. Распределение исследовательских операций

**Текущий аналитический этап:** Построение исходных тематических карт, исследование распределений показателей и их взаимосвязи.

**Последующий этап:** Оценивание моделей OLS и SAR с использованием единой пространственной матрицы весов, согласованной с другими участниками исследования.

**Примечание:** Окончательное сравнение моделей OLS, SAR и SEM, диагностика их пространственных остатков и построение итоговых карт относятся к общему командному исследованию и выполняются отдельно.

### 1.4. Ограничения интерпретации

**Наблюдательный характер данных:** Выявленная статистическая связь не доказывает причинного влияния обеспеченности врачами на продолжительность жизни.

**Пространственная неоднородность:** Регионы различаются по демографическим, социально-экономическим, климатическим и инфраструктурным условиям, вследствие чего простая парная регрессия может не учитывать существенные факторы.

**Географические ограничения:** Для двух регионов требуется дополнительная проверка территориального охвата, а историческая точность границ на 2021 год не подтверждена.

**Результат:** Формируется воспроизводимая аналитическая основа для исследования распределений показателей и последующего моделирования OLS и SAR.


In [ ]:

# ============================================================
# БЛОК 1. ПОДКЛЮЧЕНИЕ ГЕОПРОСТРАНСТВЕННЫХ ДАННЫХ
# ============================================================

# Импорт класса Path для работы с абсолютными и относительными путями.
from pathlib import Path

# Импорт NumPy для численных вычислений.
import numpy as np

# Импорт Pandas для обработки статистических показателей.
import pandas as pd

# Импорт GeoPandas для обработки пространственных геометрий.
import geopandas as gpd

# Импорт Matplotlib для построения графиков и тематических карт.
import matplotlib.pyplot as plt

# Импорт статистических функций для последующего EDA.
from scipy import stats

# Импорт функции display для отображения таблиц в Jupyter.
from IPython.display import display


# Определение необязательного пользовательского каталога данных.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analyze_spatial_models.py").is_file())
MANUAL_DIRECTORY_DATA = PROJECT_ROOT / "data" / "merged_geodata"

# Определение имени таблицы статистических показателей.
FILENAME_ATTRIBUTES = "regions_2021_attributes.csv"

# Определение имени файла географических границ.
FILENAME_GEOSPATIAL = "regions_2021_geospatial.geojson"


# ============================================================
# 1. ВЫБОР КАТАЛОГА ПРОЕКТА
# ============================================================

# ============================================================
# 2. ПОИСК СОГЛАСОВАННОЙ ПАРЫ ФАЙЛОВ
# ============================================================

# Формирование списка стандартных мест поиска данных.
directories_search = [
    Path.cwd(),
]

# Проверка наличия явно заданного пользовательского каталога.
if MANUAL_DIRECTORY_DATA is not None:

    # Замена стандартных мест поиска пользовательским расположением.
    directories_search = [
        Path(MANUAL_DIRECTORY_DATA).expanduser()
    ]

# Создание списка найденных пар статистических и географических файлов.
dataset_candidates = []

# Последовательная проверка существующих каталогов.
for directory_search in directories_search:

    # Проверка существования каталога поиска.
    if not directory_search.is_dir():

        # Переход к следующему каталогу при отсутствии текущего.
        continue

    # Поиск файлов со статистическими показателями.
    for path_attributes in directory_search.rglob(FILENAME_ATTRIBUTES):

        # Формирование ожидаемого пути к GeoJSON в том же каталоге.
        path_geospatial = path_attributes.with_name(FILENAME_GEOSPATIAL)

        # Проверка наличия согласованного географического файла.
        if path_geospatial.is_file():

            # Сохранение абсолютных путей к обоим файлам.
            dataset_candidates.append((
                path_attributes.resolve(),
                path_geospatial.resolve(),
            ))

# Удаление повторных записей одной и той же пары файлов.
dataset_candidates = list(dict.fromkeys(dataset_candidates))

# Проверка успешности поиска.
if not dataset_candidates:

    # Формирование ошибки с рекомендацией указать каталог вручную.
    raise FileNotFoundError(
        "Не найдена согласованная пара CSV и GeoJSON. "
        "Укажите каталог через MANUAL_DIRECTORY_DATA."
    )

# Проверка наличия нескольких копий набора данных.
if len(dataset_candidates) > 1:

    # Отбор файлов из стандартной папки data.
    preferred_candidates = [
        candidate
        for candidate in dataset_candidates
        if candidate[0].parent.name == "data"
    ]

    # Проверка однозначности предпочтительного расположения.
    if len(preferred_candidates) == 1:

        # Выбор согласованной пары из папки data.
        dataset_candidates = preferred_candidates

    # Обработка неоднозначного расположения данных.
    else:

        # Формирование ошибки с перечислением найденных путей.
        raise RuntimeError(
            "Найдено несколько копий данных: "
            f"{dataset_candidates}. "
            "Укажите MANUAL_DIRECTORY_DATA."
        )

# Получение путей к выбранным файлам.
path_attributes, path_geospatial = dataset_candidates[0]


# ============================================================
# 3. ЗАГРУЗКА И КОНТРОЛЬ СООТВЕТСТВИЯ
# ============================================================

# Загрузка CSV с разделителем ; и сохранением кодов как строк.
dataframe_attributes = pd.read_csv(
    path_attributes,
    sep=";",
    encoding="utf-8-sig",
    dtype={
        "object_okato": "string",
        "object_oktmo": "string",
    },
)

# Загрузка географических границ субъектов РФ.
geodataframe_boundaries = gpd.read_file(path_geospatial)

# Приведение ОКАТО статистических данных к строковому представлению.
dataframe_attributes["object_okato"] = (
    dataframe_attributes["object_okato"].astype("string").str.strip()
)

# Приведение ОКАТО географических данных к строковому представлению.
geodataframe_boundaries["object_okato"] = (
    geodataframe_boundaries["object_okato"].astype("string").str.strip()
)

# Определение количественных показателей исследования.
columns_indicators = [
    "doctors_per_10000",
    "life_expectancy_years",
]

# Проверка единого года наблюдений.
assert dataframe_attributes["year"].eq(2021).all(), (
    "Обнаружены статистические наблюдения за другой год."
)

# Проверка уникальности статистических идентификаторов ОКАТО.
assert dataframe_attributes["object_okato"].is_unique, (
    "Обнаружены повторяющиеся ОКАТО в статистике."
)

# Проверка уникальности географических идентификаторов ОКАТО.
assert geodataframe_boundaries["object_okato"].is_unique, (
    "Обнаружены повторяющиеся ОКАТО в геометрии."
)

# Проверка отсутствия пропусков в исследуемых показателях.
assert not dataframe_attributes[columns_indicators].isna().any().any(), (
    "В исходных количественных показателях обнаружены пропуски."
)

# Проверка наличия исходной системы координат.
assert geodataframe_boundaries.crs is not None, (
    "Система координат GeoJSON не определена."
)

# Проверка корректности геометрий исходных территорий.
assert geodataframe_boundaries.geometry.is_valid.all(), (
    "Исходный GeoJSON содержит невалидные геометрии."
)


# ============================================================
# 4. СОЗДАНИЕ ЕДИНОЙ АНАЛИТИЧЕСКОЙ ТАБЛИЦЫ
# ============================================================

# Объединение статистики и геометрии по однозначному ключу ОКАТО.
dataframe_merged = dataframe_attributes.merge(
    geodataframe_boundaries[["object_okato", "geometry"]],
    on="object_okato",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Проверка полного соответствия статистических и географических записей.
assert dataframe_merged["_merge"].eq("both").all(), (
    "Не для всех статистических регионов найдены геометрии."
)

# Удаление служебного столбца результата объединения.
dataframe_merged = dataframe_merged.drop(columns=["_merge"])

# Создание итогового GeoDataFrame.
geodataframe_regions = gpd.GeoDataFrame(
    dataframe_merged,
    geometry="geometry",
    crs=geodataframe_boundaries.crs,
)

# Проверка ожидаемого количества территориальных наблюдений.
assert len(geodataframe_regions) == 85, (
    "Количество регионов отличается от согласованных 85."
)

# Определение количества территорий с флагом проверки охвата.
count_regions_review = int(
    geodataframe_regions["extent_needs_review"].sum()
)

# Вывод информации об источниках данных.
print("Статистические данные:", path_attributes)
print("Географические данные:", path_geospatial)

# Вывод размерности и системы координат.
print("Количество регионов:", len(geodataframe_regions))
print("Исходная CRS:", geodataframe_regions.crs)
print("Регионов с требованием проверки охвата:", count_regions_review)

# Отображение первых пяти наблюдений без геометрического столбца.
display(
    geodataframe_regions.drop(columns="geometry").head()
)



## 2. Исходная геопространственная визуализация

### 2.1. Метод картографического представления

**Метод:** Тематическое картографирование количественных показателей по полигонам субъектов РФ (Choropleth Map).

**Первая карта:** Обеспеченность врачами всех специальностей на 10 000 человек населения.

**Вторая карта:** Ожидаемая продолжительность жизни при рождении, измеряемая в годах.

**Принцип окрашивания:** Цвет каждого региона определяется значением соответствующего статистического показателя. Непрерывная цветовая шкала позволяет сохранять различия между исходными числовыми наблюдениями.

**Картографическая проекция:** Для визуального представления используется азимутальная равновеликая проекция Ламберта с центром в точке 105° восточной долготы и 62° северной широты.

**Назначение проекции:** Формирование компактной карты пространственно протяжённой территории без непосредственного отображения долготы и широты как линейных координат.

**Примечание:** Проекция применяется к копии географических данных исключительно для картографирования. Исходные геометрии в EPSG:4326 сохраняются для дальнейшего моделирования и формирования пространственных отношений.

### 2.2. Правила интерпретации

**Интенсивность окрашивания:** Отражает величину показателя, но не статистическую значимость территориальных различий.

**Пространственные закономерности:** Группы соседних регионов со схожими значениями могут свидетельствовать о пространственной неоднородности, однако для проверки пространственной автокорреляции необходима отдельная статистическая процедура.

**Географические ограничения:** Историческая точность границ на 2021 год и охват отдельных территорий требуют дополнительной проверки.

**Результат:** Две исходные тематические карты, пригодные для визуального сопоставления пространственного распределения обеспеченности врачами и продолжительности жизни.


In [ ]:

# ============================================================
# БЛОК 2. ПОСТРОЕНИЕ ДВУХ ТЕМАТИЧЕСКИХ КАРТ
# ============================================================

# Определение параметров равновеликой проекции Ламберта.
projection_lambert = (
    "+proj=laea "
    "+lat_0=62 "
    "+lon_0=105 "
    "+datum=WGS84 "
    "+units=m "
    "+no_defs "
    "+type=crs"
)

# Создание картографической копии GeoDataFrame.
geodataframe_maps = geodataframe_regions.to_crs(
    projection_lambert
)

# Настройка шрифта, поддерживающего кириллицу.
plt.rcParams["font.family"] = "DejaVu Sans"

# Формирование описаний двух исследуемых показателей.
map_settings = [
    {
        "column": "doctors_per_10000",
        "title": "Обеспеченность врачами в субъектах РФ, 2021 год",
        "legend": "Врачей на 10 000 человек",
        "colormap": "YlGnBu",
    },
    {
        "column": "life_expectancy_years",
        "title": "Ожидаемая продолжительность жизни, 2021 год",
        "legend": "Продолжительность жизни, лет",
        "colormap": "viridis",
    },
]

# Последовательное построение отдельной карты для каждого показателя.
for map_parameters in map_settings:

    # Создание фигуры и картографических осей.
    figure, axes = plt.subplots(
        figsize=(15, 8)
    )

    # Формирование карты субъектов РФ по выбранному показателю.
    geodataframe_maps.plot(
        column=map_parameters["column"],
        cmap=map_parameters["colormap"],
        linewidth=0.16,
        edgecolor="#596574",
        legend=True,
        legend_kwds={
            "label": map_parameters["legend"],
            "shrink": 0.72,
            "format": "%.1f",
        },
        ax=axes,
    )

    # Установка названия тематической карты.
    axes.set_title(
        map_parameters["title"],
        fontsize=17,
        pad=18,
    )

    # Отключение числовых осей проекции для картографической наглядности.
    axes.set_axis_off()

    # Добавление подписи об источнике статистических данных.
    figure.text(
        0.08,
        0.035,
        "Росстат / «Если быть точным», CC BY 4.0; © OpenStreetMap contributors, ODbL 1.0; openstreetmap.org/copyright",
        fontsize=9,
        color="#555555",
    )

    # Оптимизация расположения карты и легенды.
    figure.tight_layout(rect=[0, 0.055, 1, 1])

    # Отображение итоговой тематической карты.
    plt.show()

    # Освобождение ресурсов завершённой фигуры.
    plt.close(figure)


In [ ]:

# ============================================================
# БЛОК 3. РАЗВЕДОЧНЫЙ АНАЛИЗ ИССЛЕДУЕМЫХ ПОКАЗАТЕЛЕЙ
# ============================================================

# Определение независимой переменной исследования.
series_doctors = geodataframe_regions["doctors_per_10000"]

# Определение зависимой переменной исследования.
series_life_expectancy = geodataframe_regions["life_expectancy_years"]


# ============================================================
# 1. ОПИСАТЕЛЬНАЯ СТАТИСТИКА
# ============================================================

# Расчёт описательных характеристик двух показателей.
table_descriptive_statistics = (
    geodataframe_regions[
        ["doctors_per_10000", "life_expectancy_years"]
    ]
    .describe()
    .transpose()
)

# Отбор основных статистических характеристик.
table_descriptive_statistics = table_descriptive_statistics[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

# Вывод сводной таблицы статистических характеристик.
print("Описательная статистика:")
display(table_descriptive_statistics.round(3))


# ============================================================
# 2. КОРРЕЛЯЦИОННЫЙ АНАЛИЗ
# ============================================================

# Вычисление коэффициента корреляции Пирсона.
correlation_pearson = stats.pearsonr(
    series_doctors,
    series_life_expectancy,
)

# Вычисление коэффициента ранговой корреляции Спирмена.
correlation_spearman = stats.spearmanr(
    series_doctors,
    series_life_expectancy,
)

# Формирование таблицы результатов корреляционного анализа.
table_correlations = pd.DataFrame([
    {
        "Метод": "Пирсон",
        "Коэффициент": correlation_pearson.statistic,
        "p-value": correlation_pearson.pvalue,
    },
    {
        "Метод": "Спирмен",
        "Коэффициент": correlation_spearman.statistic,
        "p-value": correlation_spearman.pvalue,
    },
])

# Вывод численных результатов.
print("Корреляционный анализ:")
display(table_correlations.round(5))


# ============================================================
# 3. СРАВНЕНИЕ ФЕДЕРАЛЬНЫХ ОКРУГОВ
# ============================================================

# Группировка регионов по федеральным округам.
table_federal_districts = (
    geodataframe_regions
    .groupby("federal_district")
    .agg(
        count_regions=("object_name", "count"),
        mean_doctors=("doctors_per_10000", "mean"),
        mean_life_expectancy=("life_expectancy_years", "mean"),
    )
    .sort_values("mean_life_expectancy", ascending=False)
)

# Вывод агрегированных характеристик федеральных округов.
print("Средние показатели по федеральным округам:")
display(table_federal_districts.round(2))


# ============================================================
# 4. ГИСТОГРАММЫ РАСПРЕДЕЛЕНИЙ
# ============================================================

# Создание двух графиков распределения.
figure_distributions, axes_distributions = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(15, 5),
)

# Построение гистограммы обеспеченности врачами.
axes_distributions[0].hist(
    series_doctors,
    bins="auto",
    color="#467DA4",
    edgecolor="white",
)

# Установка названия и подписей первой гистограммы.
axes_distributions[0].set(
    title="Распределение обеспеченности врачами",
    xlabel="Врачей на 10 000 человек",
    ylabel="Количество регионов",
)

# Построение гистограммы ожидаемой продолжительности жизни.
axes_distributions[1].hist(
    series_life_expectancy,
    bins="auto",
    color="#688F73",
    edgecolor="white",
)

# Установка названия и подписей второй гистограммы.
axes_distributions[1].set(
    title="Распределение продолжительности жизни",
    xlabel="Ожидаемая продолжительность жизни, лет",
    ylabel="Количество регионов",
)

# Оптимизация расположения графиков.
figure_distributions.tight_layout()

# Отображение распределений.
plt.show()

# Освобождение ресурсов фигуры.
plt.close(figure_distributions)


# ============================================================
# 5. ДИАГРАММА РАССЕЯНИЯ X И Y
# ============================================================

# Создание фигуры для визуального исследования взаимосвязи.
figure_scatter, axes_scatter = plt.subplots(
    figsize=(10, 7)
)

# Построение диаграммы рассеяния региональных наблюдений.
axes_scatter.scatter(
    series_doctors,
    series_life_expectancy,
    s=45,
    alpha=0.75,
    edgecolors="#303030",
    linewidths=0.35,
)

# Установка информативных названий осей и графика.
axes_scatter.set(
    title="Обеспеченность врачами и продолжительность жизни, 2021",
    xlabel="Врачей на 10 000 человек",
    ylabel="Ожидаемая продолжительность жизни, лет",
)

# Отображение вспомогательной координатной сетки.
axes_scatter.grid(
    alpha=0.25
)

# Оптимизация расположения элементов.
figure_scatter.tight_layout()

# Отображение исследовательской диаграммы.
plt.show()

# Освобождение ресурсов фигуры.
plt.close(figure_scatter)


# ============================================================
# 6. ИТОГОВЫЕ ЧИСЛЕННЫЕ ХАРАКТЕРИСТИКИ
# ============================================================

# Вывод краткого результата исследования линейной связи.
print(
    f"Корреляция Пирсона: "
    f"r = {correlation_pearson.statistic:.4f}, "
    f"p-value = {correlation_pearson.pvalue:.4f}"
)

# Вывод краткого результата исследования монотонной связи.
print(
    f"Корреляция Спирмена: "
    f"rho = {correlation_spearman.statistic:.4f}, "
    f"p-value = {correlation_spearman.pvalue:.4f}"
)

# Вывод сообщения о завершении первоначального EDA.
print(
    "Первичный разведочный анализ завершён. "
    "Данные подготовлены для последующего оценивания OLS и SAR."
)



## 3. Регрессионное моделирование продолжительности жизни

### 3.1. Базовая модель — Ordinary Least Squares (OLS)

#### Математическая постановка

**Назначение модели:** Оценивание линейной зависимости ожидаемой продолжительности жизни при рождении от обеспеченности населения врачами без явного учёта пространственных взаимодействий между регионами.

**Регрессионное уравнение:**

$$
\boxed{
y_i=\beta_0+\beta_1x_i+\varepsilon_i
}.
$$

**Определения элементов:**

- $y_i$ — ожидаемая продолжительность жизни при рождении в регионе $i$, лет.
- $x_i$ — численность врачей на 10 000 человек населения в регионе $i$.
- $\beta_0$ — свободный коэффициент регрессии.
- $\beta_1$ — коэффициент изменения ожидаемой продолжительности жизни при увеличении обеспеченности врачами на одну единицу.
- $\varepsilon_i$ — случайная ошибка модели.
- $i$ — индекс субъекта Российской Федерации.

**Матричное представление:**

$$
\boxed{
\mathbf{y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}
}.
$$

**Определения матричных объектов:**

- $\mathbf{y}\in\mathbb{R}^{n}$ — вектор зависимой переменной.
- $\mathbf{X}\in\mathbb{R}^{n\times2}$ — матрица признаков, содержащая столбец единиц и значения обеспеченности врачами.
- $\boldsymbol{\beta}\in\mathbb{R}^{2}$ — вектор неизвестных коэффициентов.
- $\boldsymbol{\varepsilon}\in\mathbb{R}^{n}$ — вектор случайных ошибок.
- $n=85$ — количество исследуемых регионов.

#### Оценивание коэффициентов

**Целевой функционал:** Сумма квадратов ошибок между фактическими и рассчитанными значениями продолжительности жизни.

$$
S(\boldsymbol{\beta})=
\sum_{i=1}^{n}
(y_i-\beta_0-\beta_1x_i)^2.
$$

**Оценка коэффициентов при полном столбцовом ранге матрицы $\mathbf{X}$:**

$$
\boxed{
\widehat{\boldsymbol{\beta}}=
(\mathbf{X}^{\mathsf{T}}\mathbf{X})^{-1}
\mathbf{X}^{\mathsf{T}}\mathbf{y}
}.
$$

**Примечание:** В программной реализации используется `statsmodels.OLS`, обеспечивающий численное оценивание коэффициентов без необходимости самостоятельно вычислять обратную матрицу.

#### Основные показатели модели

**Коэффициент детерминации:**

$$
R^2=
1-
\frac{
\sum_{i=1}^{n}(y_i-\widehat y_i)^2
}{
\sum_{i=1}^{n}(y_i-\overline y)^2
}.
$$

**Среднеквадратичная ошибка с извлечением корня:**

$$
\operatorname{RMSE}=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}(y_i-\widehat y_i)^2
}.
$$

**Дополнительные показатели:** Коэффициенты регрессии, стандартные ошибки, $p$-значения, доверительные интервалы, AIC и BIC.

**Особенность статистической интерпретации:** Обычные стандартные ошибки OLS предполагают определённые свойства ошибок модели, включая отсутствие пространственной корреляции. Робастная оценка HC3 позволяет дополнительно изучить чувствительность выводов к гетероскедастичности, но не исправляет возможную пространственную зависимость ошибок.

### 3.2. Пространственная авторегрессионная модель — SAR

#### Математическая постановка

**Назначение модели:** Оценивание взаимосвязи обеспеченности врачами и ожидаемой продолжительности жизни с учётом пространственного лага зависимой переменной.

**Модель пространственного лага:**

$$
\boxed{
\mathbf{y}=
\rho\mathbf{W}\mathbf{y}
+
\mathbf{X}\boldsymbol{\beta}
+
\boldsymbol{\varepsilon}
}.
$$

**Определения дополнительных элементов:**

- $\mathbf{W}\in\mathbb{R}^{n\times n}$ — матрица пространственных весов, описывающая связи между регионами.
- $\mathbf{W}\mathbf{y}$ — пространственный лаг ожидаемой продолжительности жизни.
- $\rho$ — коэффициент пространственной авторегрессии.
- $\mathbf{X}\boldsymbol{\beta}$ — компонента обеспеченности врачами и свободного коэффициента.
- $\boldsymbol{\varepsilon}$ — вектор случайных ошибок модели.

**Пространственный лаг отдельного региона:**

$$
(\mathbf{W}\mathbf{y})_i=
\sum_{j=1}^{n}w_{ij}y_j,
$$

где $w_{ij}$ определяет вес пространственной связи между регионами $i$ и $j$.

**Предположение:** При построчной стандартизации весов сумма ненулевых элементов каждой строки равна единице, а пространственный лаг представляет собой взвешенное среднее значений соседних регионов.

#### Приведённая форма SAR

**Преобразование уравнения:**

$$
(\mathbf{I}-\rho\mathbf{W})\mathbf{y}
=
\mathbf{X}\boldsymbol{\beta}
+
\boldsymbol{\varepsilon}.
$$

**Приведённая форма при существовании обратной матрицы:**

$$
\boxed{
\mathbf{y}=
(\mathbf{I}-\rho\mathbf{W})^{-1}
(\mathbf{X}\boldsymbol{\beta}
+\boldsymbol{\varepsilon})
}.
$$

**Определения элементов:**

- $\mathbf{I}$ — единичная матрица размерности $n\times n$.
- $(\mathbf{I}-\rho\mathbf{W})^{-1}$ — пространственный мультипликатор, учитывающий прямые и косвенные взаимодействия между регионами.

**Особенность интерпретации:** Коэффициент $\beta_1$ модели SAR нельзя автоматически трактовать как полный эффект изменения обеспеченности врачами, поскольку изменение показателя одного региона может распространяться через пространственную структуру $\mathbf{W}$.

#### Пространственные эффекты

**Матрица предельных эффектов обеспеченности врачами:**

$$
\boxed{
\mathbf{S}=
\beta_1(\mathbf{I}-\rho\mathbf{W})^{-1}
}.
$$

**Средний прямой эффект:**

$$
ADE=\frac{1}{n}\operatorname{tr}(\mathbf{S}).
$$

**Средний общий эффект:**

$$
ATE=\frac{1}{n}\mathbf{1}^{\mathsf{T}}\mathbf{S}\mathbf{1}.
$$

**Средний косвенный эффект:**

$$
AIE=ATE-ADE.
$$

**Интерпретация:** Прямой эффект характеризует средний отклик региона на изменение собственного значения обеспеченности врачами, косвенный эффект отражает пространственное распространение воздействия, а общий эффект объединяет обе составляющие.

**Примечание:** Перечисленные эффекты являются модельными ассоциациями при заданной спецификации SAR. Их причинная интерпретация требует дополнительных предпосылок.

### 3.3. Согласование пространственной структуры

**Источник матрицы:** Единая матрица пространственных весов, подготовленная в рамках общего командного исследования.

**Обязательные условия:**

1. Совпадение всех 85 территориальных идентификаторов.
2. Однозначное соответствие порядка строк матрицы весов порядку наблюдений регрессионной модели.
3. Единая пространственная схема и одинаковый способ нормирования при оценивании SAR и SEM.
4. Явное указание регионов без соседей, если они присутствуют.
5. Отсутствие автоматической замены или самостоятельного пересчёта матрицы весов при оценивании SAR.

**Метод оценивания SAR:** Максимальное правдоподобие посредством `spreg.ML_Lag`.

**Итог:** OLS формирует непространственную базовую оценку, а SAR позволяет проверить, изменяется ли описание региональной взаимосвязи при включении пространственного лага продолжительности жизни.


In [ ]:

# ============================================================
# БЛОК 4. ОЦЕНИВАНИЕ ОБЫЧНОЙ ЛИНЕЙНОЙ РЕГРЕССИИ OLS
# ============================================================

# Импорт библиотеки Statsmodels для статистического моделирования.
import statsmodels.api as sm

# Формирование независимой аналитической таблицы.
dataframe_regression = (
    geodataframe_regions[
        [
            "object_okato",
            "object_name",
            "doctors_per_10000",
            "life_expectancy_years",
        ]
    ]
    .copy()
    .sort_values("object_okato")
    .reset_index(drop=True)
)

# Проверка уникальности идентификаторов регионов.
assert dataframe_regression["object_okato"].is_unique, (
    "Идентификаторы ОКАТО должны быть уникальными."
)

# Проверка наличия всех территориальных наблюдений.
assert len(dataframe_regression) == 85, (
    "Количество регионов не соответствует исследованию."
)

# Формирование матрицы независимых переменных с константой.
matrix_predictors_ols = sm.add_constant(
    dataframe_regression[["doctors_per_10000"]],
    has_constant="add",
)

# Формирование вектора ожидаемой продолжительности жизни.
vector_target_ols = dataframe_regression[
    "life_expectancy_years"
]

# Создание спецификации обычной линейной регрессии.
regression_ols = sm.OLS(
    endog=vector_target_ols,
    exog=matrix_predictors_ols,
)

# Оценивание коэффициентов OLS.
model_ols = regression_ols.fit()

# Получение дополнительных стандартных ошибок HC3.
model_ols_hc3 = model_ols.get_robustcov_results(
    cov_type="HC3"
)

# Формирование таблицы оценённых коэффициентов.
table_parameters_ols = pd.DataFrame({
    "Параметр": matrix_predictors_ols.columns,
    "Коэффициент": np.asarray(model_ols.params),
    "Стандартная ошибка OLS": np.asarray(model_ols.bse),
    "p-value OLS": np.asarray(model_ols.pvalues),
    "Стандартная ошибка HC3": np.asarray(model_ols_hc3.bse),
    "p-value HC3": np.asarray(model_ols_hc3.pvalues),
})

# Расчёт внутривыборочной ошибки RMSE.
metric_rmse_ols = float(
    np.sqrt(np.mean(np.square(model_ols.resid)))
)

# Формирование таблицы основных характеристик модели.
table_metrics_ols = pd.DataFrame([{
    "Модель": "OLS",
    "Количество регионов": int(model_ols.nobs),
    "R2": float(model_ols.rsquared),
    "R2_adjusted": float(model_ols.rsquared_adj),
    "RMSE": metric_rmse_ols,
    "AIC": float(model_ols.aic),
    "BIC": float(model_ols.bic),
}])

# Формирование региональной таблицы с результатами OLS.
dataframe_predictions_ols = dataframe_regression[
    ["object_okato", "object_name", "life_expectancy_years"]
].copy()

# Добавление рассчитанных значений продолжительности жизни.
dataframe_predictions_ols["ols_predicted"] = (
    np.asarray(model_ols.fittedvalues, dtype=float)
)

# Добавление остатков обычной линейной регрессии.
dataframe_predictions_ols["ols_residual"] = (
    np.asarray(model_ols.resid, dtype=float)
)

# Вывод основной таблицы оценённых коэффициентов.
print("Оценки коэффициентов OLS:")
display(table_parameters_ols.round(6))

# Вывод показателей базовой регрессии.
print("Основные показатели OLS:")
display(table_metrics_ols.round(6))

# Вывод первых региональных результатов.
print("Региональные расчётные значения и остатки:")
display(dataframe_predictions_ols.head())

# Вывод полного статистического отчёта Statsmodels.
print(model_ols.summary())


Зависимости: requirements-notebook.txt.

In [ ]:

# ============================================================
# БЛОК 4. ПОДКЛЮЧЕНИЕ ОБЩЕЙ МАТРИЦЫ ПРОСТРАНСТВЕННЫХ ВЕСОВ
# ============================================================

# Импорт класса W для представления готовых пространственных весов.
from libpysal.weights import W

# Определение каталога существующей статистической базы.
directory_shared_data = PROJECT_ROOT / "results"

# Определение пути к согласованному порядку регионов.
path_weights_order = directory_shared_data / "weights_order.csv"

# Определение пути к численным пространственным весам.
path_weights_edges = directory_shared_data / "weights_edges.csv"

# Проверка наличия файлов пространственных весов.
for path_weights_file in [path_weights_order, path_weights_edges]:

    # Контроль существования обязательного файла.
    if not path_weights_file.is_file():

        # Ошибка сообщает точный путь отсутствующего файла.
        raise FileNotFoundError(
            f"Не найден файл общей матрицы: {path_weights_file}"
        )

# Загрузка порядка территорий с сохранением ОКАТО как строк.
table_weights_order = pd.read_csv(
    path_weights_order,
    sep=";",
    encoding="utf-8-sig",
    dtype={"object_okato": "string"},
)

# Загрузка готовых численных весов пространственных связей.
table_weights_edges = pd.read_csv(
    path_weights_edges,
    sep=";",
    encoding="utf-8-sig",
    dtype={
        "source_okato": "string",
        "target_okato": "string",
    },
)

# Приведение территориальных идентификаторов к единому формату.
table_weights_order["object_okato"] = (
    table_weights_order["object_okato"].str.strip().str.zfill(8)
)

# Приведение исходящих идентификаторов к единому формату.
table_weights_edges["source_okato"] = (
    table_weights_edges["source_okato"].str.strip().str.zfill(8)
)

# Приведение входящих идентификаторов к единому формату.
table_weights_edges["target_okato"] = (
    table_weights_edges["target_okato"].str.strip().str.zfill(8)
)

# Сортировка идентификаторов по исходному порядку Ильи.
table_weights_order = table_weights_order.sort_values(
    "row_index"
).reset_index(drop=True)

# Формирование упорядоченного списка ОКАТО.
list_weights_okato = table_weights_order[
    "object_okato"
].tolist()

# Проверка количества территорий в общей матрице.
assert len(list_weights_okato) == 85, (
    "Ожидается пространственная матрица для 85 регионов."
)

# Проверка уникальности территориальных идентификаторов.
assert len(set(list_weights_okato)) == 85, (
    "Обнаружены повторяющиеся ОКАТО пространственной матрицы."
)

# Проверка количества сохранённых направленных связей.
assert len(table_weights_edges) == 384, (
    "Количество связей отличается от согласованной матрицы."
)

# Проверка уникальности направленных пространственных связей.
assert not table_weights_edges.duplicated(
    subset=["source_okato", "target_okato"]
).any(), "Обнаружены повторяющиеся пространственные связи."

# Получение множества идентификаторов аналитической выборки.
set_regression_okato = set(
    dataframe_regression["object_okato"].astype(str)
)

# Проверка полного совпадения регионального состава.
assert set(list_weights_okato) == set_regression_okato, (
    "Региональный состав общей матрицы и модели не совпадает."
)

# Формирование множества идентификаторов общей матрицы.
set_weights_okato = set(list_weights_okato)

# Проверка идентификаторов исходящих пространственных связей.
assert set(
    table_weights_edges["source_okato"]
) == set_weights_okato, (
    "Состав исходящих идентификаторов не совпадает."
)

# Проверка идентификаторов входящих пространственных связей.
assert set(
    table_weights_edges["target_okato"]
).issubset(set_weights_okato), (
    "Обнаружены связи с неизвестными регионами."
)

# Формирование словаря соседей для каждой территории.
dictionary_neighbors = {
    region_okato: []
    for region_okato in list_weights_okato
}

# Формирование словаря готовых численных пространственных весов.
dictionary_weights = {
    region_okato: []
    for region_okato in list_weights_okato
}

# Последовательная обработка всех согласованных связей.
for edge in table_weights_edges.itertuples(index=False):

    # Сохранение идентификатора исходного региона.
    source_okato = str(edge.source_okato)

    # Сохранение идентификатора соседнего региона.
    target_okato = str(edge.target_okato)

    # Добавление соседа в исходную пространственную структуру.
    dictionary_neighbors[source_okato].append(
        target_okato
    )

    # Добавление исходного численного веса без изменения значения.
    dictionary_weights[source_okato].append(
        float(edge.weight)
    )

# Создание объекта W из готовых соседей и численных весов Ильи.
spatial_weights_shared = W(
    neighbors=dictionary_neighbors,
    weights=dictionary_weights,
    id_order=list_weights_okato,
    silence_warnings=True,
)

# Установка построчной стандартизации общей матрицы.
spatial_weights_shared.transform = "R"

# Получение разреженного представления пространственных весов.
matrix_weights_shared = spatial_weights_shared.sparse

# Проверка единичной суммы весов по каждой строке.
assert np.allclose(
    np.asarray(matrix_weights_shared.sum(axis=1)).ravel(),
    1.0,
), "Обнаружено нарушение построчной нормировки."

# Проверка отсутствия ненулевых весов на диагонали.
assert np.allclose(
    matrix_weights_shared.diagonal(),
    0.0,
), "Диагональ общей матрицы должна быть нулевой."

# Проверка отсутствия регионов без соседей.
assert not spatial_weights_shared.islands, (
    "Обнаружены изолированные регионы."
)

# Проверка ожидаемого числа компонент связности.
assert spatial_weights_shared.n_components == 2, (
    "Структура связности не соответствует матрице Ильи."
)

# Вывод характеристик успешно подключённой матрицы.
print("Общая матрица пространственных весов подключена.")
print("Регионов:", spatial_weights_shared.n)
print("Направленных связей:", spatial_weights_shared.nonzero)
print("Компонент связности:", spatial_weights_shared.n_components)
print("Нормирование:", spatial_weights_shared.transform)


In [ ]:

# ============================================================
# БЛОК 5. ПРОСТРАНСТВЕННАЯ АВТОРЕГРЕССИОННАЯ МОДЕЛЬ SAR
# ============================================================

# Импорт класса Path для необязательной загрузки общей матрицы весов.
from pathlib import Path

# Импорт класса ML_Lag для оценивания пространственной SAR-модели.
from spreg import ML_Lag

# Импорт библиотеки Libpysal для чтения пространственных весов.
import libpysal


# ============================================================
# 1. ПОДКЛЮЧЕНИЕ ЕДИНОЙ МАТРИЦЫ ПРОСТРАНСТВЕННЫХ ВЕСОВ
# ============================================================

# Определение необязательного пути к общей матрице в формате GAL.
PATH_SHARED_WEIGHTS = None

# Получение ранее созданной общей матрицы из пространства ноутбука.
spatial_weights_shared = globals().get(
    "spatial_weights_shared",
    None,
)

# Проверка необходимости загрузки матрицы из файла.
if spatial_weights_shared is None and PATH_SHARED_WEIGHTS is not None:

    # Формирование абсолютного пути к файлу пространственных весов.
    path_weights = Path(
        PATH_SHARED_WEIGHTS
    ).expanduser().resolve()

    # Проверка существования файла общей матрицы весов.
    if not path_weights.is_file():
        raise FileNotFoundError(
            f"Файл пространственных весов не найден: {path_weights}"
        )

    # Открытие файла пространственных весов в режиме чтения.
    reader_weights = libpysal.io.open(
        str(path_weights),
        "r",
    )

    # Чтение и гарантированное закрытие файла.
    try:

        # Загрузка общей матрицы весов.
        spatial_weights_shared = reader_weights.read()

    # Завершение работы с файловым ресурсом.
    finally:

        # Закрытие файла пространственных весов.
        reader_weights.close()


# ============================================================
# 2. ПРОВЕРКА ГОТОВНОСТИ ОБЩЕЙ МАТРИЦЫ
# ============================================================

# Проверка доступности согласованной пространственной структуры.
if spatial_weights_shared is None:

    # Информирование об ожидаемом результате командной работы.
    print(
        "SAR пока не оценена: общая матрица пространственных "
        "весов отсутствует."
    )

    # Описание допустимых способов подключения.
    print(
        "Передайте объект spatial_weights_shared "
        "либо задайте PATH_SHARED_WEIGHTS."
    )

# Выполнение SAR после подключения общей пространственной матрицы.
else:

    # Преобразование объекта Graph к формату W при необходимости.
    if hasattr(spatial_weights_shared, "to_W"):

        # Получение представления общих весов в формате W.
        spatial_weights_shared = spatial_weights_shared.to_W()

    # Проверка соответствия объекта интерфейсу библиотеки Libpysal.
    if not isinstance(spatial_weights_shared, libpysal.weights.W):
        raise TypeError(
            "Ожидается объект libpysal.weights.W "
            "или совместимый Graph."
        )

    # Проверка количества регионов в общей матрице.
    assert spatial_weights_shared.n == len(dataframe_regression), (
        "Размерность общей матрицы не соответствует числу регионов."
    )

    # Получение территориальных идентификаторов в порядке строк W.
    weights_okato_order = [
        str(region_id).strip()
        for region_id in spatial_weights_shared.id_order
    ]

    # Получение идентификаторов аналитической выборки.
    regression_okato = set(
        dataframe_regression["object_okato"].astype(str)
    )

    # Проверка уникальности идентификаторов общей матрицы.
    assert len(set(weights_okato_order)) == len(weights_okato_order), (
        "Идентификаторы пространственных весов должны быть уникальными."
    )

    # Проверка полного совпадения регионов модели и матрицы.
    assert set(weights_okato_order) == regression_okato, (
        "Состав ОКАТО общей матрицы не совпадает "
        "с регрессионной выборкой."
    )

    # Проверка заранее согласованного способа нормирования.
    assert spatial_weights_shared.transform.upper() == "R", (
        "Для SAR ожидается общая матрица W "
        "с построчной стандартизацией."
    )

    # Проверка отсутствия ненулевых собственных пространственных весов.
    assert np.allclose(
        spatial_weights_shared.sparse.diagonal(),
        0.0,
    ), "Диагональ общей матрицы W должна быть нулевой."

    # Проверка конечности числовых значений пространственных весов.
    assert np.isfinite(
        spatial_weights_shared.sparse.data
    ).all(), "Матрица W содержит некорректные числовые веса."

    # Вывод общей информации о матрице пространственных весов.
    print("Количество регионов:", spatial_weights_shared.n)
    print("Нормирование W:", spatial_weights_shared.transform)
    print("Число компонент связности:", spatial_weights_shared.n_components)
    print("Регионы без соседей:", spatial_weights_shared.islands)


    # ========================================================
    # 3. СОГЛАСОВАНИЕ ПОРЯДКА РЕГИОНОВ
    # ========================================================

    # Формирование региональной таблицы с индексом ОКАТО.
    dataframe_sar = dataframe_regression.set_index(
        "object_okato",
        drop=False,
        verify_integrity=True,
    )

    # Упорядочивание наблюдений строго по идентификаторам W.
    dataframe_sar = (
        dataframe_sar
        .loc[weights_okato_order]
        .copy()
        .reset_index(drop=True)
    )

    # Формирование матрицы независимого показателя без константы.
    matrix_predictors_sar = dataframe_sar[
        ["doctors_per_10000"]
    ].to_numpy(dtype=float)

    # Формирование столбца зависимой переменной.
    vector_target_sar = dataframe_sar[
        ["life_expectancy_years"]
    ].to_numpy(dtype=float)


    # ========================================================
    # 4. ОЦЕНИВАНИЕ ПРОСТРАНСТВЕННОЙ РЕГРЕССИИ
    # ========================================================

    # Оценивание SAR методом максимального правдоподобия.
    model_sar = ML_Lag(
        y=vector_target_sar,
        x=matrix_predictors_sar,
        w=spatial_weights_shared,
        method="full",
        vm=True,
        spat_diag=False,
        name_y="life_expectancy_years",
        name_x=["doctors_per_10000"],
        name_w="spatial_weights_shared",
        name_ds="regions_2021",
    )

    # Получение вектора оценённых параметров SAR.
    vector_parameters_sar = np.asarray(
        model_sar.betas
    ).reshape(-1)

    # Получение стандартных ошибок параметров модели.
    vector_standard_errors_sar = np.asarray(
        model_sar.std_err
    ).reshape(-1)

    # Получение Z-статистик и соответствующих p-value.
    matrix_z_statistics_sar = np.asarray(
        model_sar.z_stat,
        dtype=float,
    )

    # Формирование таблицы оценённых параметров SAR.
    table_parameters_sar = pd.DataFrame({
        "Параметр": [
            "Константа",
            "Обеспеченность врачами",
            "Пространственная авторегрессия rho",
        ],
        "Коэффициент": vector_parameters_sar,
        "Стандартная ошибка": vector_standard_errors_sar,
        "Z-статистика": matrix_z_statistics_sar[:, 0],
        "p-value": matrix_z_statistics_sar[:, 1],
    })

    # Получение рассчитанных значений из приведённой формы SAR.
    vector_predictions_sar = np.asarray(
        model_sar.predy_e
    ).reshape(-1)

    # Получение фактических значений продолжительности жизни.
    vector_actual_sar = vector_target_sar.reshape(-1)

    # Расчёт внутривыборочной ошибки приведённой формы SAR.
    metric_rmse_sar = float(
        np.sqrt(
            np.mean(
                np.square(
                    vector_actual_sar - vector_predictions_sar
                )
            )
        )
    )

    # Формирование таблицы основных характеристик SAR.
    table_metrics_sar = pd.DataFrame([{
        "Модель": "SAR",
        "Количество регионов": int(model_sar.n),
        "rho": float(model_sar.rho),
        "Pseudo_R2_reduced": float(model_sar.pr2_e),
        "RMSE_reduced": metric_rmse_sar,
        "AIC": float(model_sar.aic),
        "BIC": float(model_sar.schwarz),
    }])

    # Формирование таблицы индивидуальных результатов SAR.
    dataframe_predictions_sar = dataframe_sar[
        ["object_okato", "object_name", "life_expectancy_years"]
    ].copy()

    # Добавление рассчитанной продолжительности жизни.
    dataframe_predictions_sar["sar_predicted_reduced"] = (
        vector_predictions_sar
    )

    # Добавление ошибок прогнозирования приведённой формы.
    dataframe_predictions_sar["sar_error_reduced"] = (
        vector_actual_sar - vector_predictions_sar
    )

    # Добавление структурных остатков SAR для последующей диагностики.
    dataframe_predictions_sar["sar_residual_structural"] = (
        np.asarray(model_sar.u).reshape(-1)
    )


    # ========================================================
    # 5. ОЦЕНИВАНИЕ ПРОСТРАНСТВЕННЫХ ЭФФЕКТОВ
    # ========================================================

    # Получение оценённого коэффициента обеспеченности врачами.
    coefficient_doctors_sar = float(
        vector_parameters_sar[1]
    )

    # Получение оценённого коэффициента пространственной зависимости.
    coefficient_rho_sar = float(
        model_sar.rho
    )

    # Формирование единичной матрицы по числу регионов.
    matrix_identity = np.eye(
        spatial_weights_shared.n
    )

    # Получение матрицы общих пространственных весов.
    matrix_weights_sar = (
        spatial_weights_shared.sparse.toarray()
    )

    # Формирование матрицы пространственного мультипликатора.
    matrix_multiplier_base = (
        matrix_identity
        - coefficient_rho_sar * matrix_weights_sar
    )

    # Вычисление пространственного мультипликатора.
    matrix_spatial_multiplier = np.linalg.solve(
        matrix_multiplier_base,
        matrix_identity,
    )

    # Формирование матрицы эффектов независимой переменной.
    matrix_spatial_effects = (
        coefficient_doctors_sar * matrix_spatial_multiplier
    )

    # Вычисление среднего прямого пространственного эффекта.
    effect_direct_sar = float(
        np.trace(matrix_spatial_effects)
        / spatial_weights_shared.n
    )

    # Вычисление среднего общего пространственного эффекта.
    effect_total_sar = float(
        np.sum(matrix_spatial_effects)
        / spatial_weights_shared.n
    )

    # Вычисление среднего косвенного пространственного эффекта.
    effect_indirect_sar = (
        effect_total_sar - effect_direct_sar
    )

    # Формирование таблицы пространственных эффектов.
    table_effects_sar = pd.DataFrame([{
        "Прямой эффект": effect_direct_sar,
        "Косвенный эффект": effect_indirect_sar,
        "Общий эффект": effect_total_sar,
    }])


    # ========================================================
    # 6. ОТОБРАЖЕНИЕ РЕЗУЛЬТАТОВ SAR
    # ========================================================

    # Вывод оценённых параметров пространственной модели.
    print("Оценки коэффициентов SAR:")
    display(table_parameters_sar.round(6))

    # Вывод основных показателей пространственной регрессии.
    print("Основные показатели SAR:")
    display(table_metrics_sar.round(6))

    # Вывод оценок прямого, косвенного и общего эффектов.
    print("Пространственные эффекты обеспеченности врачами:")
    display(table_effects_sar.round(6))

    # Вывод первых региональных результатов пространственной модели.
    print("Региональные результаты SAR:")
    display(dataframe_predictions_sar.head())

    # Вывод полного статистического отчёта Spreg.
    print(model_sar.summary)



### 3.4. Результаты оценивания OLS

#### Оценённое регрессионное уравнение

**Результат вычислений по 85 субъектам РФ за 2021 год:**

$$
\boxed{
\widehat{Y}=68.9783+0.012154X
}.
$$

**Экономико-статистическая интерпретация:** Увеличению обеспеченности врачами на одного специалиста в расчёте на 10 000 человек соответствует положительное изменение рассчитанной ожидаемой продолжительности жизни приблизительно на $0.0122$ года при прочих допущениях модели.

**Показатели оценивания:**

| Характеристика | Значение |
|---|---:|
| Количество наблюдений | 85 |
| Коэффициент врачей | 0.012154 |
| $p$-значение коэффициента | 0.6313 |
| $R^2$ | 0.0028 |
| RMSE, лет | 2.2191 |
| $p$-значение коэффициента с HC3 | 0.7258 |

**Статистический вывод:** Коэффициент обеспеченности врачами не является статистически значимым на уровне $\alpha=0.05$, поскольку $p>0.05$ как при стандартном оценивании OLS, так и при использовании робастной ковариационной оценки HC3.

**Объясняющая способность:** Значение $R^2\approx0.0028$ означает, что простая регрессионная модель описывает приблизительно $0.28\%$ наблюдаемой вариации ожидаемой продолжительности жизни между регионами.

**Ограничение:** Отсутствие статистически значимой линейной зависимости в OLS не доказывает отсутствия взаимосвязи в действительности. Наблюдательные региональные данные могут характеризоваться пропущенными объясняющими факторами, пространственной зависимостью и неоднородностью условий.


### 3.5. Результаты оценивания пространственной модели SAR

#### Оценённое регрессионное уравнение

**Результат:** Пространственная авторегрессионная модель оценена методом максимального правдоподобия на 85 субъектах Российской Федерации с использованием общей матрицы весов Queen Contiguity.

$$
\boxed{
\widehat{\mathbf{y}}=
0.648511\mathbf{W}\mathbf{y}
+
23.720501
+
0.017183\mathbf{x}
}.
$$

**Примечание:** Уравнение представляет оценённую структурную спецификацию модели, а не формулу непосредственного прогноза. Для получения приведённых рассчитанных значений используется пространственный мультипликатор $(\mathbf{I}-\rho\mathbf{W})^{-1}$.

#### Оценённые параметры

| Параметр | Оценка | $p$-значение |
|---|---:|---:|
| Свободный коэффициент $\beta_0$ | 23.720501 | 0.000101 |
| Обеспеченность врачами $\beta_1$ | 0.017183 | 0.375140 |
| Пространственная авторегрессия $\rho$ | 0.648511 | $6.86\cdot10^{-14}$ |

**Пространственная зависимость:** Положительное значение $\rho$ является статистически значимым, что согласуется с наличием пространственной структуры ожидаемой продолжительности жизни при заданной модели и матрице весов.

**Обеспеченность врачами:** Коэффициент $\beta_1$ положителен, однако его $p$-значение превышает уровень значимости $0.05$, поэтому статистически значимая связь между обеспеченностью врачами и продолжительностью жизни не установлена.

#### Средние пространственные эффекты

| Характеристика | Значение |
|---|---:|
| Прямой эффект | 0.019982 |
| Косвенный эффект | 0.028903 |
| Общий эффект | 0.048886 |

**Интерпретация:** Приведённые значения характеризуют расчётные прямые и распространяющиеся через пространственные связи изменения продолжительности жизни при увеличении обеспеченности врачами на одну единицу.

**Ограничение:** Оценки пространственных эффектов приведены без доверительных интервалов и отдельной проверки статистической значимости, поэтому они не подтверждают причинного воздействия обеспеченности врачами.

#### Основные характеристики модели

| Показатель | Значение |
|---|---:|
| Количество регионов | 85 |
| AIC | 350.962 |
| BIC | 358.290 |
| RMSE приведённой формы, лет | 2.279 |

**Результат сопоставления с OLS:** Пространственная регрессия SAR имеет меньшее значение AIC, чем обычная линейная регрессия, однако несколько большее значение внутривыборочной RMSE.

**Вывод:** Учёт пространственного лага зависимой переменной существенно изменяет статистическое описание региональной продолжительности жизни. При этом независимый показатель обеспеченности врачами остаётся статистически незначимым.

### 3.6. Передача результатов для общего командного исследования

**Объекты OLS:**

- `model_ols` — оценённая обычная линейная регрессия.
- `table_parameters_ols` — коэффициенты и стандартные ошибки.
- `table_metrics_ols` — основные количественные показатели.
- `dataframe_predictions_ols` — региональные расчётные значения и остатки.

**Объекты SAR:**

- `model_sar` — оценённая пространственная авторегрессионная модель.
- `table_parameters_sar` — коэффициенты, стандартные ошибки и $p$-значения.
- `table_metrics_sar` — основные характеристики модели.
- `table_effects_sar` — оценки прямого, косвенного и общего пространственных эффектов.
- `dataframe_predictions_sar` — региональные рассчитанные значения и структурные остатки.

**Принцип интеграции:** Таблицы индивидуальных результатов содержат уникальный код ОКАТО, позволяющий однозначно присоединять рассчитанные показатели к географическим границам.

**Разграничение качества:** Коэффициент детерминации $R^2$ обычной модели и псевдокоэффициент $\operatorname{Pseudo}R^2$ пространственной модели имеют различные определения, поэтому их значения нельзя трактовать как полностью эквивалентные показатели.

**Примечание:** Окончательные сравнение OLS, SAR и SEM, диагностика пространственной автокорреляции остатков, выбор итоговой модели и построение результирующих карт выполняются в общем командном исследовании.

### 3.7. Итоговый исследовательский вывод

**Результат OLS:** Простая линейная регрессия не выявила статистически значимой связи между обеспеченностью врачами и ожидаемой продолжительностью жизни ($p=0.6313$).

**Результат SAR:** Пространственный авторегрессионный коэффициент оказался положительным и статистически значимым ($\rho=0.6485$, $p<0.001$), тогда как обеспеченность врачами сохранила статистическую незначимость ($p=0.3751$).

**Сопоставление моделей:** Значение AIC уменьшилось с $380.729$ для OLS до $350.962$ для SAR, что указывает на преимущество пространственной модели по данному информационному критерию. Внутривыборочная RMSE SAR при этом несколько выше, чем у OLS.

**Ограничения:** Пространственные взаимодействия являются модельными статистическими зависимостями, а не доказательством причинного влияния обеспеченности врачами на продолжительность жизни.

**Итог:** Оценены базовая регрессия OLS и пространственная модель SAR на согласованных данных 85 регионов за 2021 год. Результаты подготовлены для общего сравнения с SEM, диагностики и построения итоговых карт в рамках командного исследования.
